# 04 - Genetic Algorithm


## Idea general

Un algoritmo genetico es una metaheuristica poblacional inspirada en la evolucion
natural.

La idea central es mantener un conjunto de soluciones candidatas, llamado
poblacion, y hacerlo evolucionar durante varias generaciones. Las soluciones con
mejor desempeno tienen mas probabilidad de influir en la siguiente generacion,
pero tambien se introduce variacion para seguir explorando.

En este metodo, una sola solucion no recorre el espacio por si misma. La busqueda
se reparte entre muchos individuos. Eso permite explorar varias zonas al mismo
tiempo.

Como toda metaheuristica, no garantiza encontrar el optimo global. Busca buenas
soluciones combinando seleccion, recombinacion y cambio aleatorio controlado.


## Individuo y poblacion

Un individuo es una solucion candidata del problema.

La poblacion es el conjunto de individuos que el algoritmo mantiene en una
generacion.

La forma de representar un individuo se llama codificacion. En TSP puede ser una
permutacion de ciudades. En mochila puede ser un vector binario. En optimizacion
continua puede ser un vector de numeros reales.

La codificacion es una decision importante, porque define que operadores tienen
sentido. No se cruza igual una permutacion que un vector binario o uno continuo.


## Fitness

El fitness mide que tan bueno es un individuo.

En un algoritmo genetico normalmente se habla de maximizar fitness. Si el
problema original es de minimizacion, se transforma el objetivo para que una
mejor solucion tenga mayor fitness. Por ejemplo, si se minimiza costo, se puede
usar fitness negativo del costo.

El fitness no es solo una medida de calidad. Tambien guia la seleccion: los
individuos con mejor fitness tienen mas probabilidad de reproducirse o pasar
informacion a la siguiente generacion.


## Seleccion

La seleccion decide que individuos se usan como padres.

El objetivo es dar mas oportunidades a las soluciones buenas, sin eliminar por
completo la diversidad. Si la seleccion es demasiado fuerte, la poblacion puede
converger muy rapido a una zona mediocre. Si es demasiado debil, el algoritmo
avanza sin direccion.

Una seleccion comun es torneo. Se eligen algunos individuos al azar y gana el de
mejor fitness dentro de ese grupo.

La seleccion produce presion evolutiva: empuja la poblacion hacia soluciones mas
prometedoras.


## Crossover

El crossover, o cruce, combina informacion de dos padres para generar nuevos
hijos.

La idea es que dos soluciones buenas pueden contener partes utiles, y que al
recombinarlas podria aparecer una solucion mejor.

El tipo de crossover depende de la codificacion. En vectores binarios se pueden
intercambiar segmentos. En variables reales se pueden mezclar valores. En
permutaciones, como rutas, se necesitan cruces especiales para no repetir ni
perder elementos.

Crossover es principalmente una herramienta de explotacion: usa informacion de
soluciones buenas que ya existen.


## Mutacion

La mutacion introduce cambios aleatorios pequenos en un individuo.

Su funcion principal es mantener diversidad y evitar que toda la poblacion se
vuelva demasiado parecida.

En un vector binario, mutar puede ser cambiar un 0 por 1. En una permutacion,
puede ser intercambiar dos posiciones o invertir un segmento. En variables
continuas, puede ser sumar ruido.

Mutacion es principalmente una herramienta de exploracion: permite visitar zonas
que no aparecen solo combinando padres.



La diversidad significa que la poblacion no esta formada por copias casi iguales.
Es importante porque una poblacion diversa mantiene varias alternativas vivas.
Si todos los individuos se parecen demasiado pronto, el algoritmo puede quedar
encerrado en una zona mediocre.

Ese problema se llama convergencia prematura. No significa que el algoritmo
fallo por error de codigo, sino que la poblacion perdio variedad antes de
explorar suficiente.


## Elitismo

El elitismo copia algunos de los mejores individuos directamente a la siguiente
generacion.

Esto evita perder la mejor solucion encontrada por culpa de un cruce o una
mutacion desfavorable.

Un poco de elitismo suele ser util. Demasiado elitismo puede reducir diversidad
y hacer que la poblacion se estanque.

La idea completa queda asi: seleccion y elitismo empujan hacia lo que ya se ve
bueno; crossover combina informacion; mutacion evita que la poblacion se cierre
demasiado rapido. El equilibrio entre esas fuerzas es lo que hace funcionar al
metodo.


## Estructura en Python

La forma tecnica de implementar un algoritmo genetico general es separar el motor
evolutivo del problema especifico.

En el notebook, la estructura se implementa con estas piezas:

- `initial_population`: poblacion inicial.
- `fitness(individual)`: mide la calidad del individuo.
- `crossover(parent1, parent2, rng)`: genera hijos combinando padres.
- `mutate(individual, rng)`: modifica un individuo.
- `generations`: cantidad de generaciones.
- `elite_size`: numero de mejores individuos que pasan directo.
- `crossover_rate`: probabilidad de aplicar cruce.
- `tournament_size`: intensidad de la seleccion por torneo.

El motor genetico no necesita saber si el individuo representa una ruta, una
mochila o un vector real. Eso queda definido por la codificacion y los
operadores que se le entregan.


In [1]:
from copy import deepcopy
from dataclasses import dataclass
import random

@dataclass
class GeneticAlgorithmResult:
    best_individual: object
    best_fitness: float
    history: list
    final_population: list
    generations: int


def tournament_selection(population, fitness_values, tournament_size, rng):
    size = min(tournament_size, len(population))
    competitors = rng.sample(range(len(population)), size)
    winner = max(competitors, key=lambda i: fitness_values[i])
    return deepcopy(population[winner])


def genetic_algorithm(
    initial_population,
    fitness,
    crossover,
    mutate,
    *,
    generations=100,
    elite_size=1,
    crossover_rate=0.9,
    tournament_size=3,
    seed=None,
):
    """
    Algoritmo genetico generico.

    Este motor maximiza fitness. Para problemas de minimizacion, se puede usar
    una transformacion como fitness = -objective.
    """
    if not initial_population:
        raise ValueError("initial_population no puede estar vacia")

    rng = random.Random(seed)
    population = [deepcopy(individual) for individual in initial_population]
    population_size = len(population)
    elite_size = max(0, min(elite_size, population_size))

    fitness_values = [fitness(individual) for individual in population]
    best_index = max(range(population_size), key=lambda i: fitness_values[i])
    best_individual = deepcopy(population[best_index])
    best_fitness = fitness_values[best_index]
    history = [best_fitness]

    for _ in range(generations):
        ranked = sorted(
            zip(fitness_values, population),
            key=lambda pair: pair[0],
            reverse=True,
        )

        next_population = [deepcopy(individual) for _, individual in ranked[:elite_size]]

        while len(next_population) < population_size:
            parent1 = tournament_selection(population, fitness_values, tournament_size, rng)
            parent2 = tournament_selection(population, fitness_values, tournament_size, rng)

            if rng.random() < crossover_rate:
                child1, child2 = crossover(parent1, parent2, rng)
            else:
                child1, child2 = deepcopy(parent1), deepcopy(parent2)

            next_population.append(mutate(child1, rng))
            if len(next_population) < population_size:
                next_population.append(mutate(child2, rng))

        population = next_population
        fitness_values = [fitness(individual) for individual in population]

        generation_best = max(range(population_size), key=lambda i: fitness_values[i])
        if fitness_values[generation_best] > best_fitness:
            best_individual = deepcopy(population[generation_best])
            best_fitness = fitness_values[generation_best]

        history.append(best_fitness)

    return GeneticAlgorithmResult(
        best_individual=best_individual,
        best_fitness=best_fitness,
        history=history,
        final_population=population,
        generations=generations,
    )

## Lectura del codigo

El codigo implementa un algoritmo genetico general que maximiza fitness.

La clase `GeneticAlgorithmResult` guarda la mejor solucion encontrada, su
fitness, el historial de mejora, la poblacion final y el numero de generaciones.

La funcion `tournament_selection` realiza la seleccion por torneo. Toma algunos
individuos al azar y devuelve una copia del que tiene mejor fitness dentro de
ese grupo.

La funcion `genetic_algorithm` recibe una poblacion inicial y las funciones que
definen el problema: fitness, crossover y mutacion. Con eso puede trabajar sobre
distintas codificaciones.

En cada generacion, primero se evalua el fitness de toda la poblacion. Luego se
copian los mejores individuos como elites.

Despues se crean hijos hasta completar la nueva poblacion. Para eso se eligen
padres por torneo, se aplica crossover con cierta probabilidad y finalmente se
aplica mutacion.

Al terminar la generacion, la nueva poblacion reemplaza a la anterior. Si aparece
un individuo con mejor fitness que el mejor conocido, se actualiza el
incumbente.

La idea importante es que el algoritmo no mejora una solucion aislada, sino que
hace evolucionar una poblacion completa.


## Ejemplo 1 - TSP

En TSP, un individuo es una permutacion de ciudades. La poblacion contiene muchas rutas candidatas.

El crossover debe cuidar que el hijo siga siendo una permutacion valida, sin ciudades repetidas ni ciudades faltantes.


In [2]:
import math
import random

coords = [
    (0.10, 0.20), (0.25, 0.85), (0.50, 0.55),
    (0.80, 0.75), (0.90, 0.15), (0.45, 0.10),
    (0.15, 0.55), (0.65, 0.25),
]

n = len(coords)

def distance(a, b):
    ax, ay = coords[a]
    bx, by = coords[b]
    return math.hypot(ax - bx, ay - by)

def tour_length(tour):
    return sum(distance(tour[i], tour[(i + 1) % len(tour)]) for i in range(len(tour)))

def tsp_fitness(tour):
    return -tour_length(tour)

def order_crossover(parent1, parent2, rng):
    a, b = sorted(rng.sample(range(n), 2))
    def build(base, donor):
        child = [None] * n
        child[a:b + 1] = base[a:b + 1]
        used = set(child[a:b + 1])
        pos = (b + 1) % n
        for city in donor[b + 1:] + donor[:b + 1]:
            if city not in used:
                child[pos] = city
                pos = (pos + 1) % n
        return tuple(child)
    return build(parent1, parent2), build(parent2, parent1)

def inversion_mutation(tour, rng, rate=0.25):
    tour = list(tour)
    if rng.random() < rate:
        a, b = sorted(rng.sample(range(n), 2))
        tour[a:b + 1] = reversed(tour[a:b + 1])
    return tuple(tour)

rng = random.Random(11)
initial_population = [tuple(rng.sample(range(n), n)) for _ in range(60)]

result = genetic_algorithm(
    initial_population,
    tsp_fitness,
    order_crossover,
    inversion_mutation,
    generations=120,
    elite_size=3,
    crossover_rate=0.9,
    tournament_size=3,
    seed=5,
)

print("Mejor tour:", result.best_individual)
print("Distancia:", round(-result.best_fitness, 4))
print("Fitness inicial mejor:", round(result.history[0], 4))
print("Fitness final:", round(result.history[-1], 4))

Mejor tour: (0, 5, 7, 4, 3, 2, 1, 6)
Distancia: 2.9124
Fitness inicial mejor: -3.3406
Fitness final: -2.9124


## Lectura del ejemplo TSP

El fitness se define como el negativo de la distancia, porque el motor genetico maximiza fitness pero TSP minimiza distancia.

La mutacion invierte un segmento de la ruta. Esto mantiene una permutacion valida y agrega diversidad a la poblacion.


## Ejemplo 2 - Mochila 0/1

La mochila 0/1 es natural para algoritmos geneticos porque una solucion puede representarse como un cromosoma binario.

Cada gen indica si un item se toma o no. El fitness premia valor alto y penaliza soluciones que superan la capacidad.


In [3]:
items = [
    ("A", 20, 2), ("B", 30, 5), ("C", 35, 7),
    ("D", 12, 3), ("E", 3, 1), ("F", 50, 9),
    ("G", 25, 6), ("H", 18, 4),
]
capacity = 15
n_items = len(items)

def chromosome_value(chromosome):
    return sum(value for bit, (_, value, _) in zip(chromosome, items) if bit)

def chromosome_weight(chromosome):
    return sum(weight for bit, (_, _, weight) in zip(chromosome, items) if bit)

def knapsack_fitness(chromosome):
    value = chromosome_value(chromosome)
    weight = chromosome_weight(chromosome)
    overweight = max(0, weight - capacity)
    return value - 1000 * overweight

def one_point_crossover(parent1, parent2, rng):
    cut = rng.randint(1, n_items - 1)
    child1 = parent1[:cut] + parent2[cut:]
    child2 = parent2[:cut] + parent1[cut:]
    return child1, child2

def bit_flip_mutation(chromosome, rng, rate=0.08):
    genes = list(chromosome)
    for i in range(len(genes)):
        if rng.random() < rate:
            genes[i] = 1 - genes[i]
    return tuple(genes)

rng = random.Random(13)
initial_population = [tuple(rng.randint(0, 1) for _ in range(n_items)) for _ in range(80)]

result = genetic_algorithm(
    initial_population,
    knapsack_fitness,
    one_point_crossover,
    bit_flip_mutation,
    generations=100,
    elite_size=4,
    crossover_rate=0.85,
    tournament_size=4,
    seed=9,
)

chosen = [name for bit, (name, _, _) in zip(result.best_individual, items) if bit]
print("Cromosoma:", result.best_individual)
print("Items elegidos:", chosen)
print("Valor:", chromosome_value(result.best_individual))
print("Peso:", chromosome_weight(result.best_individual))
print("Fitness:", result.best_fitness)

Cromosoma: (1, 1, 1, 0, 1, 0, 0, 0)
Items elegidos: ['A', 'B', 'C', 'E']
Valor: 88
Peso: 15
Fitness: 88


## Lectura del ejemplo Mochila

El crossover mezcla partes de dos cromosomas. La mutacion cambia bits para explorar combinaciones nuevas.

Este ejemplo muestra por que la codificacion es central en algoritmos geneticos: el motor es el mismo, pero los operadores dependen del tipo de solucion.


## Parametros

Los parametros principales son tamano de poblacion, generaciones, tasa de cruce,
tasa de mutacion, tamano del torneo y elitismo.

Una poblacion mas grande explora mas, pero cuesta mas evaluar. Mas generaciones
dan mas tiempo para mejorar, pero aumentan el costo total.

Una tasa de cruce alta favorece recombinacion. Una tasa de mutacion alta aumenta
exploracion, pero si es excesiva puede convertir la busqueda en algo casi
aleatorio.

El tamano del torneo controla la presion selectiva. Torneos grandes hacen que
ganen casi siempre los mejores, pero pueden reducir diversidad demasiado rapido.


## Complejidad

El costo principal esta en evaluar la poblacion.

De forma general, el costo depende del tamano de la poblacion, el numero de
generaciones y el costo de calcular el fitness de cada individuo.

Si el fitness es caro, el algoritmo genetico tambien lo sera. Por eso muchas
implementaciones intentan paralelizar evaluaciones o reducir calculos repetidos.


## Cuando se usa

Los algoritmos geneticos se usan cuando existen muchas soluciones posibles y se
puede definir una buena codificacion junto con operadores de cruce y mutacion.

Son utiles en problemas combinatorios, seleccion de subconjuntos, rutas,
calibracion de parametros, diseno y optimizacion continua.

Funcionan especialmente bien cuando combinar partes de buenas soluciones puede
generar soluciones aun mejores.


## Resumen

Un algoritmo genetico es una metaheuristica poblacional que hace evolucionar una
poblacion de soluciones.

Sus ideas centrales son fitness, seleccion, crossover, mutacion y elitismo.

La seleccion y el crossover explotan soluciones prometedoras. La mutacion
mantiene exploracion y diversidad. El elitismo protege las mejores soluciones.

Su fortaleza es la flexibilidad: puede adaptarse a muchos problemas cambiando la
codificacion y los operadores. Su debilidad es que depende mucho de parametros y
puede converger prematuramente si pierde diversidad.
